# A decode-attention kernel in Triton: paged KV cache, GQA and split-KV

Write the kernel that dominates long-context decode: one query token per sequence attending over a KV cache stored in scattered pages, with grouped-query attention, online softmax across pages, and a split-KV second stage that spreads long sequences over many SMs. Includes a reference implementation, tests on ragged batches and a bandwidth target to measure against.

This notebook holds the code from the lesson **[A decode-attention kernel in Triton: paged KV cache, GQA and split-KV](https://inferenceacademy.com/learn/kern-triton/a-decode-attention-kernel-in-triton)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## The problem

## What speed to aim for

## Stage 1: one program per (sequence, head, split)

In [ ]:
import torch, triton, triton.language as tl

@triton.jit
def decode_attn_stage1(q_ptr, k_ptr, v_ptr, bt_ptr, lens_ptr, o_ptr, lse_ptr,
                       sm_scale, stride_bt, n_splits,
                       stride_qb, stride_qh,
                       stride_kp, stride_kt, stride_kh,          # cache: [pages, PAGE, H_kv, D]
                       stride_ob, stride_oh, stride_os,          # o: [B, H, n_splits, D]
                       GROUP: tl.constexpr, PAGE: tl.constexpr, D: tl.constexpr):
    b, h, s = tl.program_id(0), tl.program_id(1), tl.program_id(2)
    kvh = h // GROUP                                   # GQA: which KV head this query head reads
    L = tl.load(lens_ptr + b)
    chunk = tl.cdiv(tl.cdiv(L, n_splits), PAGE) * PAGE # tokens per split, whole pages
    start = s * chunk
    end = tl.minimum(start + chunk, L)

    d = tl.arange(0, D)
    t = tl.arange(0, PAGE)
    q = tl.load(q_ptr + b * stride_qb + h * stride_qh + d).to(tl.float32)
    m = -float("inf")                                  # running max
    l = 0.0                                            # running sum of exponentials
    acc = tl.zeros([D], dtype=tl.float32)              # running weighted sum of V

    for pos in range(start, end, PAGE):
        page = tl.load(bt_ptr + b * stride_bt + pos // PAGE)       # logical page -> physical page
        valid = pos + t < end
        off = page * stride_kp + t[:, None] * stride_kt + kvh * stride_kh + d[None, :]
        k = tl.load(k_ptr + off, mask=valid[:, None], other=0.0).to(tl.float32)   # [PAGE, D]
        scores = tl.sum(k * q[None, :], axis=1) * sm_scale                       # [PAGE]
        scores = tl.where(valid, scores, -float("inf"))
        m_new = tl.maximum(m, tl.max(scores, axis=0))
        alpha = tl.exp(m - m_new)                      # rescales everything accumulated so far
        p = tl.exp(scores - m_new)
        v = tl.load(v_ptr + off, mask=valid[:, None], other=0.0).to(tl.float32)
        acc = acc * alpha + tl.sum(p[:, None] * v, axis=0)
        l = l * alpha + tl.sum(p, axis=0)
        m = m_new

    o = acc / tl.where(l > 0, l, 1.0)                  # an empty split stores zeros
    tl.store(o_ptr + b * stride_ob + h * stride_oh + s * stride_os + d, o)
    tl.store(lse_ptr + (b * tl.num_programs(1) + h) * n_splits + s, m + tl.log(l))

## Stage 2: merging the splits

In [ ]:
@triton.jit
def decode_attn_stage2(o_ptr, lse_ptr, out_ptr, n_splits,
                       stride_ob, stride_oh, stride_os, stride_outb, stride_outh,
                       D: tl.constexpr, SPLITS: tl.constexpr):
    b, h = tl.program_id(0), tl.program_id(1)
    s = tl.arange(0, SPLITS)
    smask = s < n_splits
    lse = tl.load(lse_ptr + (b * tl.num_programs(1) + h) * n_splits + s, mask=smask, other=-float("inf"))
    w = tl.exp(lse - tl.max(lse, axis=0))              # empty splits get weight 0
    d = tl.arange(0, D)
    o = tl.load(o_ptr + b * stride_ob + h * stride_oh + s[:, None] * stride_os + d[None, :],
                mask=smask[:, None], other=0.0)
    out = tl.sum(w[:, None] * o, axis=0) / tl.sum(w, axis=0)
    tl.store(out_ptr + b * stride_outb + h * stride_outh + d, out.to(out_ptr.dtype.element_ty))

## The wrapper

In [ ]:
def paged_decode_attention(q, k_cache, v_cache, block_table, seq_lens, n_splits=8):
    B, H, D = q.shape
    _, PAGE, H_kv, _ = k_cache.shape
    assert k_cache.stride() == v_cache.stride() and k_cache.stride(3) == 1 and q.stride(2) == 1
    o = torch.empty(B, H, n_splits, D, device=q.device, dtype=torch.float32)
    lse = torch.empty(B, H, n_splits, device=q.device, dtype=torch.float32)
    decode_attn_stage1[(B, H, n_splits)](
        q, k_cache, v_cache, block_table, seq_lens, o, lse,
        D ** -0.5, block_table.stride(0), n_splits,
        q.stride(0), q.stride(1),
        k_cache.stride(0), k_cache.stride(1), k_cache.stride(2),
        o.stride(0), o.stride(1), o.stride(2),
        GROUP=H // H_kv, PAGE=PAGE, D=D)
    out = torch.empty_like(q)
    decode_attn_stage2[(B, H)](o, lse, out, n_splits, o.stride(0), o.stride(1), o.stride(2),
                               out.stride(0), out.stride(1), D=D, SPLITS=triton.next_power_of_2(n_splits))
    return out

## Testing on a ragged, scattered batch

In [ ]:
def reference(q, k_cache, v_cache, block_table, seq_lens):
    B, H, D = q.shape
    PAGE, H_kv = k_cache.shape[1], k_cache.shape[2]
    out = torch.empty_like(q)
    for b in range(B):
        L = int(seq_lens[b])
        pages = block_table[b, : (L + PAGE - 1) // PAGE].long()
        k = k_cache[pages].reshape(-1, H_kv, D)[:L].repeat_interleave(H // H_kv, dim=1).float()
        v = v_cache[pages].reshape(-1, H_kv, D)[:L].repeat_interleave(H // H_kv, dim=1).float()
        scores = torch.einsum("hd,lhd->hl", q[b].float(), k) * D ** -0.5
        out[b] = torch.einsum("hl,lhd->hd", scores.softmax(-1), v).to(q.dtype)
    return out

B, H, H_kv, D, PAGE = 4, 32, 8, 128, 16
seq_lens = torch.tensor([1, 100, 1000, 4096], device="cuda", dtype=torch.int32)
max_pages = (int(seq_lens.max()) + PAGE - 1) // PAGE
k_cache = torch.randn(B * max_pages, PAGE, H_kv, D, device="cuda", dtype=torch.float16)
v_cache = torch.randn_like(k_cache)
block_table = torch.randperm(B * max_pages, device="cuda").to(torch.int32).view(B, max_pages)  # scattered pages
q = torch.randn(B, H, D, device="cuda", dtype=torch.float16)

for splits in [1, 2, 8, 32]:
    got = paged_decode_attention(q, k_cache, v_cache, block_table, seq_lens, n_splits=splits)
    torch.testing.assert_close(got, reference(q, k_cache, v_cache, block_table, seq_lens), atol=2e-3, rtol=2e-3)
print("ok")

## Measuring it

In [ ]:
B, L = 32, 4096
seq_lens = torch.full((B,), L, device="cuda", dtype=torch.int32)
max_pages = L // PAGE
k_cache = torch.randn(B * max_pages, PAGE, H_kv, D, device="cuda", dtype=torch.float16)
v_cache = torch.randn_like(k_cache)
block_table = torch.randperm(B * max_pages, device="cuda").to(torch.int32).view(B, max_pages)
q = torch.randn(B, H, D, device="cuda", dtype=torch.float16)
kv_bytes = 2 * k_cache.numel() * k_cache.element_size()
for splits in [1, 4, 16]:
    ms = triton.testing.do_bench(lambda: paged_decode_attention(q, k_cache, v_cache, block_table, seq_lens, splits))
    print(f"splits={splits:2d}: {ms:.3f} ms, {kv_bytes / ms / 1e6:.0f} GB/s")

## What this kernel leaves out, and what real ones add

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-triton/a-decode-attention-kernel-in-triton